In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np


pd.set_option('display.max_colwidth', None)

ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

DATA = ROOT / "data" / "prb_articles_labeled_Jan2016-Jun2026_duplicate-free.json"
DATA_SAMPLE = ROOT / "data" / "prb_articles_labeled_Jan2016-Jun2026_duplicate-free_sample.json"

#CLEANED_DATA is output of prep_data() from preprocessing.py
CLEANED_DATA  = ROOT / "data" / "cleaned_data.json"

MAPPER_MODULE_PATH = str(ROOT / "src" / "data")

#For importing functions from preprocessing.py
if MAPPER_MODULE_PATH not in sys.path:
    sys.path.append(MAPPER_MODULE_PATH)

data_to_load = DATA if DATA.exists() else DATA_SAMPLE
df = pd.read_json(CLEANED_DATA)

print(f"Loaded data from {data_to_load.name}")
df.info()

Loaded data from prb_articles_labeled_Jan2016-Jun2026_duplicate-free.json
<class 'pandas.DataFrame'>
RangeIndex: 50398 entries, 0 to 50397
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   doi          50398 non-null  str           
 1   title        50398 non-null  str           
 2   abstract     50398 non-null  str           
 3   physh        50398 non-null  object        
 4   date         50398 non-null  datetime64[us]
 5   articleType  50398 non-null  str           
 6   physh_names  50398 non-null  object        
dtypes: datetime64[us](1), object(2), str(4)
memory usage: 2.7+ MB


## PhysBERT Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load PhysBERT tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("thellert/physbert_uncased")
model_physbert = AutoModel.from_pretrained("thellert/physbert_uncased")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [49]:
from torch.utils.data import DataLoader

batch_size = 25
dataset_loader = DataLoader(df['abstract'].to_list(), batch_size=batch_size)


In [ ]:
# Get one batch
first_batch = next(iter(dataset_loader))

# Tokenize the input text and pass it through the model
inputs = tokenizer(first_batch, padding=True, 
    truncation=True,return_tensors='pt')
outputs = model_physhbert(**inputs)


# Extract the token embeddings
token_embeddings = outputs.last_hidden_state
# Drop CLS and SEP tokens, then take the mean for the sentence embedding
token_embeddings = token_embeddings[:, 1:-1, :] #careful here
sentence_embedding = token_embeddings.mean(dim=1)

## napkinXC

In [66]:
from sklearn.preprocessing import MultiLabelBinarizer
from napkinxc.models import PLT

In [64]:
mlb = MultiLabelBinarizer()
y_train_binarized_physbert = mlb.fit_transform(df['physh_names'].head(25))


In [76]:
# Load Probabilistic Label Tree if already exists, train if doesn't
napkinxc_physbert_path = Path("physh_plt_model_physbert")

if (napkinxc_physbert_path / "weights.bin").exists():
    napkinxc_physbert_model = PLT(str(napkinxc_physbert_path))

else:
    napkinxc_physbert_model = PLT(str(napkinxc_physbert_path))
    napkinxc_physbert_model.fit(sentence_embedding.detach().numpy(), y_train_binarized_physbert)



In [79]:

# Sample text to embed
prediction_text = "In this study, we propose that the signatures of spin fractionalization in quantum magnets can be identified through a detailed analysis of the temperature dependence of the asymmetric Fano lineshape of optical phonons overlapping with a continuum of spin excitations. We focus on the hyperhoneycomb magnet 𝛽−Li2⁢IrO3, a promising candidate for being in proximity to a three-dimensional Kitaev quantum spin liquid. The Raman response in 𝛽−Li2⁢IrO3 notably displays a distinctive asymmetric Fano lineshape in the 24 meV Raman-active optical phonon. This asymmetry arises from the interaction between the discrete phonon mode and the spin excitation continuum, which could be fractionalized if the material is indeed near a quantum spin-liquid phase. Our theoretical model considers the coupling of this optical phonon to Majorana fermions in the Kitaev model on the hyperhoneycomb lattice. Our findings reveal that the temperature-dependent Fano lineshape is consistent with the fractionalization of spins into Majorana fermions and ℤ2 fluxes."

# Tokenize the input text and pass it through the model
inputs = tokenizer(sample_text, return_tensors="pt")
outputs = model_physhbert(**inputs)

# Extract the token embeddings
token_embeddings = outputs.last_hidden_state
# Drop CLS and SEP tokens, then take the mean for the sentence embedding
token_embeddings = token_embeddings[:, 1:-1, :]
sentence_embedding_prediction = token_embeddings.mean(dim=1)


In [ ]:
#Invoke model to make predictions on a given article
napkinxc_physbert_model.predict(sentence_embedding_prediction.detach().numpy(), top_k=5)

[[75, 24, 95, 86, 22]]

In [96]:
[[mlb.classes_[idx] for idx in sample] for sample in [[75, 24, 95, 86, 22]]]

[['Quantum transport',
  'First-principles calculations',
  'Topological materials',
  'Superconductivity',
  'Excitons']]

In [ ]:
print(precision_at_k(y_test_binarized_preprune, predictions_napkinXC_preprune, k=5)) 


[0.56200397 0.48685516 0.43511905 0.39637897 0.36603175]
